In [ ]:
import io
import tkinter as tk
from tkinter import filedialog, messagebox

import cv2
import numpy as np

from PIL import Image, ImageTk
import pillow_avif


class ChaosImageApp:

    def __init__(self, root):
        self.root = root
        self.root.title("Liquify Chaos Lab")

        # =========================================================
        # OPEN IMAGE
        # =========================================================

        path = filedialog.askopenfilename(
            title="Выберите изображение",
            filetypes=[
                (
                    "Images",
                    "*.avif *.png *.jpg *.jpeg *.webp",
                ),
                ("AVIF", "*.avif"),
                ("PNG", "*.png"),
                ("JPEG", "*.jpg *.jpeg"),
                ("WEBP", "*.webp"),
                ("All files", "*.*"),
            ],
        )

        if not path:
            root.destroy()
            return

        try:
            self.original = Image.open(path).convert("RGB")
        except Exception as e:
            messagebox.showerror(
                "Ошибка",
                f"Не удалось открыть изображение:\n\n{e}",
            )
            root.destroy()
            return

        self.original_np = np.asarray(
            self.original,
            dtype=np.uint8,
        )

        self.original_h, self.original_w = (
            self.original_np.shape[:2]
        )

        # =========================================================
        # PREVIEW SIZE
        # =========================================================

        max_width = 1200
        max_height = 760

        self.scale = min(
            max_width / self.original_w,
            max_height / self.original_h,
            1.0,
        )

        self.work_w = max(
            1,
            int(self.original_w * self.scale),
        )

        self.work_h = max(
            1,
            int(self.original_h * self.scale),
        )

        resized = self.original.resize(
            (self.work_w, self.work_h),
            Image.Resampling.LANCZOS,
        )

        # Базовое изображение никогда не меняется
        self.base_np = np.asarray(
            resized,
            dtype=np.uint8,
        )

        # =========================================================
        # LIQUIFY MAP
        # =========================================================

        self.disp_x = np.zeros(
            (self.work_h, self.work_w),
            dtype=np.float32,
        )

        self.disp_y = np.zeros(
            (self.work_h, self.work_w),
            dtype=np.float32,
        )

        # =========================================================
        # BRUSH SETTINGS
        # =========================================================

        self.radius = 60.0
        self.strength = 0.8

        self.last_x = None
        self.last_y = None

        # =========================================================
        # CHAOS SETTINGS
        # =========================================================

        self.feedback = 0.45
        self.feedback_steps = 4

        self.fold = 0.15
        self.twist = 0.15

        self.echo_shift = 5.0
        self.echo_decay = 0.78

        # =========================================================
        # TEXTURE / CONVOLUTION
        # =========================================================

        self.texture_kernel = None
        self.texture_path = None

        self.texture_size = 21
        self.texture_strength = 0.35

        self.texture_kernel_var = tk.DoubleVar(
            value=self.texture_size
        )

        self.texture_strength_var = tk.DoubleVar(
            value=self.texture_strength
        )

        # =========================================================
        # CANVAS
        # =========================================================

        self.canvas = tk.Canvas(
            root,
            width=self.work_w,
            height=self.work_h,
            bg="black",
            highlightthickness=0,
        )

        self.canvas.pack()

        self.tk_image = None

        self.image_item = self.canvas.create_image(
            0,
            0,
            anchor="nw",
        )

        # =========================================================
        # BRUSH CIRCLE
        # =========================================================

        self.brush_circle = self.canvas.create_oval(
            0,
            0,
            0,
            0,
            outline="white",
            width=2,
        )

        # =========================================================
        # CONTROL PANEL
        # =========================================================

        controls = tk.Frame(root)
        controls.pack(
            fill="x",
            padx=5,
            pady=5,
        )

        # =========================================================
        # BUTTONS
        # =========================================================

        button_row = tk.Frame(controls)
        button_row.pack(
            fill="x",
            pady=(0, 5),
        )

        tk.Button(
            button_row,
            text="LOAD TEXTURE",
            command=self.load_texture,
        ).pack(
            side="left",
            padx=2,
        )

        tk.Button(
            button_row,
            text="SAVE",
            command=self.save,
        ).pack(
            side="left",
            padx=2,
        )

        tk.Button(
            button_row,
            text="RESET",
            command=self.reset,
        ).pack(
            side="left",
            padx=2,
        )

        tk.Button(
            button_row,
            text="RANDOM DIRT",
            command=self.random_chaos,
        ).pack(
            side="left",
            padx=2,
        )

        tk.Button(
            button_row,
            text="NEW KERNEL",
            command=self.randomize_kernel,
        ).pack(
            side="left",
            padx=2,
        )

        # =========================================================
        # INFO
        # =========================================================

        self.info = tk.Label(
            button_row,
            text="",
            anchor="w",
        )

        self.info.pack(
            side="left",
            padx=10,
        )

        # =========================================================
        # SLIDERS
        # =========================================================

        self.add_slider(
            controls,
            "Brush size",
            self.radius,
            5,
            500,
            1,
            self.change_radius,
        )

        self.add_slider(
            controls,
            "Brush strength",
            self.strength,
            0.05,
            2.0,
            0.05,
            self.change_strength,
        )

        self.add_slider(
            controls,
            "Feedback",
            self.feedback,
            0.0,
            0.95,
            0.01,
            self.change_feedback,
        )

        self.add_slider(
            controls,
            "Iterations",
            self.feedback_steps,
            1,
            12,
            1,
            self.change_steps,
        )

        self.add_slider(
            controls,
            "Fold",
            self.fold,
            0.0,
            1.0,
            0.01,
            self.change_fold,
        )

        self.add_slider(
            controls,
            "Twist",
            self.twist,
            -2.0,
            2.0,
            0.01,
            self.change_twist,
        )

        self.add_slider(
            controls,
            "Echo",
            self.echo_shift,
            0,
            50,
            1,
            self.change_echo,
        )

        self.add_slider(
            controls,
            "Texture size",
            self.texture_size,
            3,
            51,
            2,
            self.change_texture_size,
        )

        self.add_slider(
            controls,
            "Texture strength",
            self.texture_strength,
            0.0,
            2.0,
            0.01,
            self.change_texture_strength,
        )

        # =========================================================
        # MOUSE EVENTS
        # =========================================================

        self.canvas.bind(
            "<ButtonPress-1>",
            self.mouse_down,
        )

        self.canvas.bind(
            "<B1-Motion>",
            self.mouse_drag,
        )

        self.canvas.bind(
            "<ButtonRelease-1>",
            self.mouse_up,
        )

        self.canvas.bind(
            "<Motion>",
            self.mouse_move,
        )

        self.canvas.bind(
            "<MouseWheel>",
            self.mouse_wheel,
        )

        # Linux
        self.canvas.bind(
            "<Button-4>",
            lambda event: self.change_radius(10),
        )

        self.canvas.bind(
            "<Button-5>",
            lambda event: self.change_radius(-10),
        )

        # =========================================================
        # KEYS
        # =========================================================

        root.bind(
            "[",
            lambda event: self.change_radius(-10),
        )

        root.bind(
            "]",
            lambda event: self.change_radius(10),
        )

        root.bind(
            "r",
            lambda event: self.reset(),
        )

        root.bind(
            "c",
            lambda event: self.random_chaos(),
        )

        # =========================================================
        # FIRST RENDER
        # =========================================================

        self.update_info()
        self.refresh_image()

    # =============================================================
    # SLIDER
    # =============================================================

    def add_slider(
        self,
        parent,
        name,
        value,
        minimum,
        maximum,
        resolution,
        callback,
    ):
        frame = tk.Frame(parent)
        frame.pack(
            fill="x",
        )

        label = tk.Label(
            frame,
            text=name,
            width=18,
            anchor="w",
        )

        label.pack(
            side="left",
        )

        scale = tk.Scale(
            frame,
            from_=minimum,
            to=maximum,
            resolution=resolution,
            orient="horizontal",
            showvalue=True,
            command=callback,
        )

        scale.set(value)

        scale.pack(
            side="left",
            fill="x",
            expand=True,
        )

    # =============================================================
    # SETTINGS
    # =============================================================

    def change_radius(self, value):
        self.radius += float(value)

        self.radius = max(
            5,
            min(
                500,
                self.radius,
            ),
        )

        self.update_info()

    def change_strength(self, value):
        self.strength = float(value)
        self.update_info()

    def change_feedback(self, value):
        self.feedback = float(value)
        self.refresh_image()

    def change_steps(self, value):
        self.feedback_steps = max(
            1,
            int(float(value)),
        )
        self.refresh_image()

    def change_fold(self, value):
        self.fold = float(value)
        self.refresh_image()

    def change_twist(self, value):
        self.twist = float(value)
        self.refresh_image()

    def change_echo(self, value):
        self.echo_shift = float(value)
        self.refresh_image()

    def change_texture_size(self, value):
        size = int(float(value))

        if size % 2 == 0:
            size += 1

        self.texture_size = max(
            3,
            min(
                51,
                size,
            ),
        )

        # Если текстура уже загружена,
        # пересоздаём kernel.
        if self.texture_path is not None:
            self.build_texture_kernel(
                self.texture_path
            )

        self.refresh_image()

    def change_texture_strength(self, value):
        self.texture_strength = float(value)
        self.refresh_image()

    # =============================================================
    # INFO
    # =============================================================

    def update_info(self):
        texture_name = "none"

        if self.texture_path is not None:
            texture_name = "loaded"

        self.info.config(
            text=(
                f"Brush: {int(self.radius)}   "
                f"Texture: {texture_name}"
            )
        )

    # =============================================================
    # LOAD TEXTURE
    # =============================================================

    def load_texture(self):
        path = filedialog.askopenfilename(
            title="Выберите текстуру",
            filetypes=[
                (
                    "Images",
                    "*.avif *.png *.jpg *.jpeg *.webp *.svg",
                ),
                ("AVIF", "*.avif"),
                ("PNG", "*.png"),
                ("JPEG", "*.jpg *.jpeg"),
                ("WEBP", "*.webp"),
                ("SVG", "*.svg"),
                ("All files", "*.*"),
            ],
        )

        if not path:
            return

        try:
            self.build_texture_kernel(path)

            self.texture_path = path

            self.update_info()
            self.refresh_image()

            messagebox.showinfo(
                "Текстура загружена",
                (
                    f"Текстура успешно загружена.\n\n"
                    f"Kernel: {self.texture_size} × "
                    f"{self.texture_size}"
                ),
            )

        except Exception as e:
            messagebox.showerror(
                "Ошибка загрузки текстуры",
                f"{e}",
            )

    # =============================================================
    # BUILD TEXTURE KERNEL
    # =============================================================

    def build_texture_kernel(self, path):

        size = int(self.texture_size)

        if size % 2 == 0:
            size += 1

        # ---------------------------------------------------------
        # SVG
        # ---------------------------------------------------------

        if path.lower().endswith(".svg"):
            import cairosvg

            png_data = cairosvg.svg2png(
                url=path,
                output_width=size,
                output_height=size,
            )

            texture = Image.open(
                io.BytesIO(png_data)
            ).convert("L")

        # ---------------------------------------------------------
        # RASTER
        # ---------------------------------------------------------

        else:
            texture = Image.open(path).convert("L")

        texture = texture.resize(
            (size, size),
            Image.Resampling.LANCZOS,
        )

        texture_np = np.asarray(
            texture,
            dtype=np.float32,
        )

        # ---------------------------------------------------------
        # Убираем DC component
        # ---------------------------------------------------------

        texture_np -= texture_np.mean()

        # ---------------------------------------------------------
        # Нормализация std
        # ---------------------------------------------------------

        std = texture_np.std()

        if std > 1e-6:
            texture_np /= std

        # ---------------------------------------------------------
        # L1 normalization
        # ---------------------------------------------------------

        absolute_sum = np.sum(
            np.abs(texture_np)
        )

        if absolute_sum > 1e-6:
            texture_np /= absolute_sum

        # ---------------------------------------------------------
        # Корректируем направление для convolution
        # ---------------------------------------------------------

        texture_np = np.flip(
            texture_np,
            axis=(0, 1),
        ).copy()

        self.texture_kernel = texture_np

    # =============================================================
    # RANDOM KERNEL
    # =============================================================

    def randomize_kernel(self):

        size = int(self.texture_size)

        if size % 2 == 0:
            size += 1

        kernel = np.random.normal(
            0,
            1,
            (size, size),
        ).astype(np.float32)

        # Делаем немного более выраженный центр
        center = size // 2

        kernel[center, center] += 1.5

        # Убираем среднюю составляющую
        kernel -= kernel.mean()

        absolute_sum = np.sum(
            np.abs(kernel)
        )

        if absolute_sum > 1e-6:
            kernel /= absolute_sum

        self.texture_kernel = kernel
        self.texture_path = "RANDOM"

        self.update_info()
        self.refresh_image()

    # =============================================================
    # TEXTURE CONVOLUTION
    # =============================================================

    def apply_texture_convolution(
        self,
        image,
    ):
        if self.texture_kernel is None:
            return image

        kernel = (
            self.texture_kernel
            * self.texture_strength
        )

        # ---------------------------------------------------------
        # RGB -> convolution
        # ---------------------------------------------------------

        image_float = image.astype(
            np.float32
        )

        filtered = cv2.filter2D(
            image_float,
            ddepth=cv2.CV_32F,
            kernel=kernel,
            borderType=cv2.BORDER_REFLECT,
        )

        # ---------------------------------------------------------
        # Добавляем detail обратно в изображение.
        #
        # Это стабильнее, чем просто заменять изображение
        # результатом convolution.
        # ---------------------------------------------------------

        result = (
            image_float
            + filtered
        )

        result = np.clip(
            result,
            0,
            255,
        )

        return result.astype(
            np.uint8
        )

    # =============================================================
    # LIQUIFY
    # =============================================================

    def render_liquify(self):

        h, w = self.base_np.shape[:2]

        y, x = np.indices(
            (h, w),
            dtype=np.float32,
        )

        map_x = (
            x
            - self.disp_x
        )

        map_y = (
            y
            - self.disp_y
        )

        map_x = np.clip(
            map_x,
            0,
            w - 1,
        )

        map_y = np.clip(
            map_y,
            0,
            h - 1,
        )

        result = cv2.remap(
            self.base_np,
            map_x,
            map_y,
            interpolation=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT,
        )

        return result

    # =============================================================
    # CHAOS ENGINE
    # =============================================================

    def render_chaos(self):

        current = self.render_liquify()

        h, w = current.shape[:2]

        y, x = np.indices(
            (h, w),
            dtype=np.float32,
        )

        cx = w / 2.0
        cy = h / 2.0

        rx = x - cx
        ry = y - cy

        distance = np.sqrt(
            rx * rx
            + ry * ry
        )

        max_radius = np.sqrt(
            cx * cx
            + cy * cy
        )

        normalized_radius = (
            distance
            / max_radius
        )

        # =========================================================
        # FEEDBACK LOOP
        # =========================================================

        for i in range(
            self.feedback_steps
        ):

            # -----------------------------------------------------
            # FOLD
            # -----------------------------------------------------

            period = max(
                30.0,
                min(h, w) * 0.35,
            )

            folded_x = (
                (
                    rx
                    + period / 2.0
                )
                % period
            ) - period / 2.0

            folded_y = (
                (
                    ry
                    + period / 2.0
                )
                % period
            ) - period / 2.0

            folded_x = np.abs(
                folded_x
            )

            folded_y = np.abs(
                folded_y
            )

            fold_mask = (
                np.abs(
                    np.sin(
                        normalized_radius
                        * np.pi
                    )
                )
            )

            fold_strength = (
                self.fold
                * fold_mask
            )

            fx = (
                rx
                * (1.0 - fold_strength)
                +
                folded_x
                * fold_strength
            )

            fy = (
                ry
                * (1.0 - fold_strength)
                +
                folded_y
                * fold_strength
            )

            # -----------------------------------------------------
            # TWIST
            # -----------------------------------------------------

            angle = (
                self.twist
                * np.sin(
                    normalized_radius
                    * np.pi
                    * 2.0
                )
                * (
                    1.0
                    -
                    (
                        i
                        /
                        max(
                            1,
                            self.feedback_steps,
                        )
                    )
                )
            )

            cos_a = np.cos(angle)
            sin_a = np.sin(angle)

            tx = (
                fx * cos_a
                -
                fy * sin_a
            )

            ty = (
                fx * sin_a
                +
                fy * cos_a
            )

            # -----------------------------------------------------
            # ECHO
            # -----------------------------------------------------

            direction = (
                1.0
                if i % 2 == 0
                else -1.0
            )

            echo_x = (
                direction
                * self.echo_shift
                * (i + 1)
            )

            echo_y = (
                -direction
                * self.echo_shift
                * 0.5
                * (i + 1)
            )

            src_x = (
                cx
                + tx
                + echo_x
            )

            src_y = (
                cy
                + ty
                + echo_y
            )

            src_x = np.clip(
                src_x,
                0,
                w - 1,
            )

            src_y = np.clip(
                src_y,
                0,
                h - 1,
            )

            warped = cv2.remap(
                current,
                src_x.astype(
                    np.float32
                ),
                src_y.astype(
                    np.float32
                ),
                interpolation=cv2.INTER_LINEAR,
                borderMode=cv2.BORDER_REFLECT,
            )

            # -----------------------------------------------------
            # FEEDBACK MIX
            # -----------------------------------------------------

            alpha = (
                self.feedback
                * (
                    self.echo_decay
                    ** i
                )
            )

            alpha = np.clip(
                alpha,
                0.0,
                0.95,
            )

            current = (
                current.astype(
                    np.float32
                )
                * (1.0 - alpha)
                +
                warped.astype(
                    np.float32
                )
                * alpha
            )

            current = np.clip(
                current,
                0,
                255,
            ).astype(
                np.uint8
            )

            # -----------------------------------------------------
            # TEXTURE CONVOLUTION
            # -----------------------------------------------------

            if self.texture_kernel is not None:
                current = (
                    self.apply_texture_convolution(
                        current
                    )
                )

        return current

    # =============================================================
    # REFRESH
    # =============================================================

    def refresh_image(self):

        result = self.render_chaos()

        pil_image = Image.fromarray(
            result
        )

        self.tk_image = ImageTk.PhotoImage(
            pil_image
        )

        self.canvas.itemconfig(
            self.image_item,
            image=self.tk_image,
        )

        self.canvas.tag_raise(
            self.brush_circle
        )

    # =============================================================
    # MOUSE POSITION
    # =============================================================

    def mouse_position(self, event):

        x = max(
            0,
            min(
                self.work_w - 1,
                event.x,
            ),
        )

        y = max(
            0,
            min(
                self.work_h - 1,
                event.y,
            ),
        )

        return (
            float(x),
            float(y),
        )

    # =============================================================
    # BRUSH STAMP
    # =============================================================

    def apply_brush(
        self,
        center_x,
        center_y,
        move_x,
        move_y,
    ):

        radius = float(
            self.radius
        )

        x0 = max(
            0,
            int(
                center_x - radius
            ),
        )

        x1 = min(
            self.work_w,
            int(
                center_x
                + radius
                + 1
            ),
        )

        y0 = max(
            0,
            int(
                center_y - radius
            ),
        )

        y1 = min(
            self.work_h,
            int(
                center_y
                + radius
                + 1
            ),
        )

        if x0 >= x1 or y0 >= y1:
            return

        yy, xx = np.mgrid[
            y0:y1,
            x0:x1,
        ]

        dx = (
            xx
            - center_x
        )

        dy = (
            yy
            - center_y
        )

        distance = np.sqrt(
            dx * dx
            + dy * dy
        )

        mask = (
            distance
            <= radius
        )

        t = np.clip(
            1.0
            - distance / radius,
            0.0,
            1.0,
        )

        # Smoothstep
        weight = (
            t
            * t
            * (
                3.0
                - 2.0 * t
            )
        )

        weight *= mask
        weight *= self.strength

        self.disp_x[
            y0:y1,
            x0:x1,
        ] += (
            move_x
            * weight
        )

        self.disp_y[
            y0:y1,
            x0:x1,
        ] += (
            move_y
            * weight
        )

    # =============================================================
    # BRUSH STROKE
    # =============================================================

    def apply_stroke(
        self,
        x0,
        y0,
        x1,
        y1,
    ):

        dx = x1 - x0
        dy = y1 - y0

        distance = np.sqrt(
            dx * dx
            + dy * dy
        )

        if distance < 0.01:
            return

        step_size = max(
            2.0,
            self.radius * 0.15,
        )

        steps = max(
            1,
            int(
                distance
                /
                step_size
            ),
        )

        for i in range(
            1,
            steps + 1
        ):

            t = i / steps

            px = (
                x0
                + dx * t
            )

            py = (
                y0
                + dy * t
            )

            self.apply_brush(
                px,
                py,
                dx / steps,
                dy / steps,
            )

    # =============================================================
    # MOUSE DOWN
    # =============================================================

    def mouse_down(self, event):

        (
            self.last_x,
            self.last_y,
        ) = self.mouse_position(
            event
        )

    # =============================================================
    # MOUSE DRAG
    # =============================================================

    def mouse_drag(self, event):

        x, y = self.mouse_position(
            event
        )

        if (
            self.last_x is not None
            and self.last_y is not None
        ):

            self.apply_stroke(
                self.last_x,
                self.last_y,
                x,
                y,
            )

            self.refresh_image()

        self.last_x = x
        self.last_y = y

    # =============================================================
    # MOUSE UP
    # =============================================================

    def mouse_up(self, event):

        self.last_x = None
        self.last_y = None

    # =============================================================
    # MOUSE MOVE
    # =============================================================

    def mouse_move(self, event):

        x, y = self.mouse_position(
            event
        )

        r = self.radius

        self.canvas.coords(
            self.brush_circle,
            x - r,
            y - r,
            x + r,
            y + r,
        )

        self.canvas.tag_raise(
            self.brush_circle
        )

    # =============================================================
    # MOUSE WHEEL
    # =============================================================

    def mouse_wheel(self, event):

        if event.delta > 0:
            self.change_radius(10)
        else:
            self.change_radius(-10)

    # =============================================================
    # RANDOM CHAOS
    # =============================================================

    def random_chaos(self):

        self.feedback = np.random.uniform(
            0.25,
            0.90,
        )

        self.feedback_steps = np.random.randint(
            2,
            9,
        )

        self.fold = np.random.uniform(
            0.0,
            0.95,
        )

        self.twist = np.random.uniform(
            -1.5,
            1.5,
        )

        self.echo_shift = np.random.uniform(
            0,
            35,
        )

        self.strength = np.random.uniform(
            0.3,
            1.5,
        )

        self.texture_strength = np.random.uniform(
            0.0,
            1.0,
        )

        self.update_info()
        self.refresh_image()

    # =============================================================
    # RESET
    # =============================================================

    def reset(self):

        self.disp_x.fill(0)
        self.disp_y.fill(0)

        self.feedback = 0.45
        self.feedback_steps = 4
        self.fold = 0.15
        self.twist = 0.15
        self.echo_shift = 5.0

        self.strength = 0.8
        self.texture_strength = 0.35

        self.update_info()
        self.refresh_image()

    # =============================================================
    # SAVE
    # =============================================================

    def save(self):

        output = filedialog.asksaveasfilename(
            title="Сохранить",
            defaultextension=".avif",
            filetypes=[
                ("AVIF", "*.avif"),
                ("PNG", "*.png"),
                ("JPEG", "*.jpg"),
            ],
        )

        if not output:
            return

        try:

            # Рендерим текущее состояние
            result = self.render_chaos()

            # Возвращаем размер к исходному
            if (
                result.shape[1]
                != self.original_w
                or
                result.shape[0]
                != self.original_h
            ):

                result_image = Image.fromarray(
                    result
                )

                result_image = result_image.resize(
                    (
                        self.original_w,
                        self.original_h,
                    ),
                    Image.Resampling.LANCZOS,
                )

            else:

                result_image = Image.fromarray(
                    result
                )

            lower = output.lower()

            # -----------------------------------------------------
            # AVIF
            # -----------------------------------------------------

            if lower.endswith(".avif"):

                result_image.save(
                    output,
                    format="AVIF",
                    quality=90,
                )

            # -----------------------------------------------------
            # JPEG
            # -----------------------------------------------------

            elif (
                lower.endswith(".jpg")
                or lower.endswith(".jpeg")
            ):

                result_image.save(
                    output,
                    format="JPEG",
                    quality=95,
                )

            # -----------------------------------------------------
            # PNG
            # -----------------------------------------------------

            else:

                result_image.save(
                    output,
                    format="PNG",
                )

            messagebox.showinfo(
                "Готово",
                f"Изображение сохранено:\n\n{output}",
            )

        except Exception as e:

            messagebox.showerror(
                "Ошибка сохранения",
                str(e),
            )


# ================================================================
# RUN
# ================================================================

if __name__ == "__main__":

    root = tk.Tk()

    app = ChaosImageApp(root)

    root.mainloop()

In [1]:
import io
import tkinter as tk
from tkinter import filedialog, messagebox

import cv2
import numpy as np

from PIL import Image, ImageTk
import pillow_avif


class ChaosImageApp:

    def __init__(self, root):
        self.root = root
        self.root.title("Liquify Chaos Lab")

        # =========================================================
        # РАЗМЕР ЭКРАНА
        # =========================================================

        screen_w = root.winfo_screenwidth()
        screen_h = root.winfo_screenheight()

        self.root.geometry(
            f"{int(screen_w * 0.92)}x{int(screen_h * 0.90)}"
        )

        # =========================================================
        # ОТКРЫТИЕ ИЗОБРАЖЕНИЯ
        # =========================================================

        path = filedialog.askopenfilename(
            title="Выберите изображение",
            filetypes=[
                (
                    "Images",
                    "*.avif *.png *.jpg *.jpeg *.webp"
                ),
                ("AVIF", "*.avif"),
                ("PNG", "*.png"),
                ("JPEG", "*.jpg *.jpeg"),
                ("WEBP", "*.webp"),
                ("All files", "*.*"),
            ],
        )

        if not path:
            root.destroy()
            return

        try:
            self.original = Image.open(path).convert("RGB")
        except Exception as e:
            messagebox.showerror(
                "Ошибка",
                f"Не удалось открыть изображение:\n\n{e}"
            )
            root.destroy()
            return

        self.original_np = np.asarray(
            self.original,
            dtype=np.uint8
        )

        self.original_h, self.original_w = (
            self.original_np.shape[:2]
        )

        # =========================================================
        # РАЗМЕР PREVIEW
        # =========================================================

        # Оставляем место справа под UI
        max_width = int(screen_w * 0.62)
        max_height = int(screen_h * 0.78)

        self.scale = min(
            max_width / self.original_w,
            max_height / self.original_h,
            1.0
        )

        self.work_w = max(
            1,
            int(self.original_w * self.scale)
        )

        self.work_h = max(
            1,
            int(self.original_h * self.scale)
        )

        resized = self.original.resize(
            (self.work_w, self.work_h),
            Image.Resampling.LANCZOS
        )

        self.base_np = np.asarray(
            resized,
            dtype=np.uint8
        )

        # =========================================================
        # LIQUIFY MAP
        # =========================================================

        self.disp_x = np.zeros(
            (self.work_h, self.work_w),
            dtype=np.float32
        )

        self.disp_y = np.zeros(
            (self.work_h, self.work_w),
            dtype=np.float32
        )

        # =========================================================
        # BRUSH
        # =========================================================

        self.radius = 60.0
        self.strength = 0.8

        self.last_x = None
        self.last_y = None

        # =========================================================
        # CHAOS
        # =========================================================

        self.feedback = 0.45
        self.feedback_steps = 4
        self.fold = 0.15
        self.twist = 0.15
        self.echo_shift = 5.0
        self.echo_decay = 0.78

        # =========================================================
        # TEXTURE
        # =========================================================

        self.texture_kernel = None
        self.texture_path = None

        self.texture_size = 21
        self.texture_strength = 0.35

        # =========================================================
        # ОСНОВНОЙ LAYOUT
        # =========================================================

        main_frame = tk.Frame(root)
        main_frame.pack(
            fill="both",
            expand=True
        )

        # =========================================================
        # ЛЕВАЯ ЧАСТЬ
        # =========================================================

        image_frame = tk.Frame(
            main_frame
        )

        image_frame.pack(
            side="left",
            fill="both",
            expand=True
        )

        # =========================================================
        # ПРАВАЯ ПАНЕЛЬ
        # =========================================================

        controls_outer = tk.Frame(
            main_frame,
            width=360
        )

        controls_outer.pack(
            side="right",
            fill="y"
        )

        controls_outer.pack_propagate(False)

        # =========================================================
        # CANVAS ИЗОБРАЖЕНИЯ
        # =========================================================

        self.canvas = tk.Canvas(
            image_frame,
            bg="black",
            highlightthickness=0
        )

        self.canvas.pack(
            fill="both",
            expand=True,
            padx=5,
            pady=5
        )

        # =========================================================
        # SCROLLABLE PANEL
        # =========================================================

        controls_canvas = tk.Canvas(
            controls_outer,
            highlightthickness=0
        )

        scrollbar = tk.Scrollbar(
            controls_outer,
            orient="vertical",
            command=controls_canvas.yview
        )

        controls_canvas.configure(
            yscrollcommand=scrollbar.set
        )

        scrollbar.pack(
            side="right",
            fill="y"
        )

        controls_canvas.pack(
            side="left",
            fill="both",
            expand=True
        )

        controls = tk.Frame(
            controls_canvas
        )

        controls_window = controls_canvas.create_window(
            (0, 0),
            window=controls,
            anchor="nw"
        )

        def update_scroll_region(event=None):
            controls_canvas.configure(
                scrollregion=controls_canvas.bbox("all")
            )

        def resize_controls(event):
            controls_canvas.itemconfig(
                controls_window,
                width=event.width
            )

        controls.bind(
            "<Configure>",
            update_scroll_region
        )

        controls_canvas.bind(
            "<Configure>",
            resize_controls
        )

        # =========================================================
        # SCROLL ПАНЕЛИ
        # =========================================================

        def scroll_controls(event):
            controls_canvas.yview_scroll(
                int(-event.delta / 120),
                "units"
            )

        controls_canvas.bind(
            "<MouseWheel>",
            scroll_controls
        )

        # Linux
        controls_canvas.bind(
            "<Button-4>",
            lambda event: controls_canvas.yview_scroll(
                -1,
                "units"
            )
        )

        controls_canvas.bind(
            "<Button-5>",
            lambda event: controls_canvas.yview_scroll(
                1,
                "units"
            )
        )

        # =========================================================
        # HEADER
        # =========================================================

        title = tk.Label(
            controls,
            text="LIQUIFY CHAOS",
            font=("Arial", 18, "bold")
        )

        title.pack(
            pady=(12, 4)
        )

        subtitle = tk.Label(
            controls,
            text="Image destruction playground",
            fg="gray"
        )

        subtitle.pack(
            pady=(0, 12)
        )

        # =========================================================
        # BUTTONS
        # =========================================================

        button_frame = tk.Frame(
            controls
        )

        button_frame.pack(
            fill="x",
            padx=10
        )

        tk.Button(
            button_frame,
            text="LOAD TEXTURE",
            command=self.load_texture
        ).pack(
            fill="x",
            pady=2
        )

        tk.Button(
            button_frame,
            text="NEW RANDOM KERNEL",
            command=self.randomize_kernel
        ).pack(
            fill="x",
            pady=2
        )

        tk.Button(
            button_frame,
            text="RANDOM DIRT",
            command=self.random_chaos
        ).pack(
            fill="x",
            pady=2
        )

        tk.Button(
            button_frame,
            text="RESET",
            command=self.reset
        ).pack(
            fill="x",
            pady=2
        )

        tk.Button(
            button_frame,
            text="SAVE",
            command=self.save
        ).pack(
            fill="x",
            pady=2
        )

        # =========================================================
        # INFO
        # =========================================================

        self.info = tk.Label(
            controls,
            text="",
            justify="left",
            anchor="w"
        )

        self.info.pack(
            fill="x",
            padx=10,
            pady=10
        )

        # =========================================================
        # СЕКЦИЯ LIQUIFY
        # =========================================================

        self.add_section(
            controls,
            "LIQUIFY"
        )

        self.add_slider(
            controls,
            "Brush size",
            self.radius,
            5,
            500,
            1,
            self.change_radius
        )

        self.add_slider(
            controls,
            "Brush strength",
            self.strength,
            0.05,
            2.0,
            0.05,
            self.change_strength
        )

        # =========================================================
        # FEEDBACK
        # =========================================================

        self.add_section(
            controls,
            "FEEDBACK"
        )

        self.add_slider(
            controls,
            "Feedback",
            self.feedback,
            0.0,
            0.95,
            0.01,
            self.change_feedback
        )

        self.add_slider(
            controls,
            "Iterations",
            self.feedback_steps,
            1,
            15,
            1,
            self.change_steps
        )

        self.add_slider(
            controls,
            "Echo",
            self.echo_shift,
            0,
            60,
            1,
            self.change_echo
        )

        # =========================================================
        # FOLD / TWIST
        # =========================================================

        self.add_section(
            controls,
            "FOLD / TWIST"
        )

        self.add_slider(
            controls,
            "Fold",
            self.fold,
            0.0,
            1.0,
            0.01,
            self.change_fold
        )

        self.add_slider(
            controls,
            "Twist",
            self.twist,
            -2.0,
            2.0,
            0.01,
            self.change_twist
        )

        # =========================================================
        # TEXTURE
        # =========================================================

        self.add_section(
            controls,
            "TEXTURE CONVOLUTION"
        )

        self.add_slider(
            controls,
            "Kernel size",
            self.texture_size,
            3,
            51,
            2,
            self.change_texture_size
        )

        self.add_slider(
            controls,
            "Texture strength",
            self.texture_strength,
            0.0,
            2.0,
            0.01,
            self.change_texture_strength
        )

        # =========================================================
        # MOUSE
        # =========================================================

        self.canvas.bind(
            "<ButtonPress-1>",
            self.mouse_down
        )

        self.canvas.bind(
            "<B1-Motion>",
            self.mouse_drag
        )

        self.canvas.bind(
            "<ButtonRelease-1>",
            self.mouse_up
        )

        self.canvas.bind(
            "<Motion>",
            self.mouse_move
        )

        self.canvas.bind(
            "<MouseWheel>",
            self.mouse_wheel
        )

        self.canvas.bind(
            "<Button-4>",
            lambda event: self.change_radius(10)
        )

        self.canvas.bind(
            "<Button-5>",
            lambda event: self.change_radius(-10)
        )

        # =========================================================
        # KEYBOARD
        # =========================================================

        root.bind(
            "[",
            lambda event: self.change_radius(-10)
        )

        root.bind(
            "]",
            lambda event: self.change_radius(10)
        )

        root.bind(
            "r",
            lambda event: self.reset()
        )

        root.bind(
            "c",
            lambda event: self.random_chaos()
        )

        # =========================================================
        # IMAGE
        # =========================================================

        self.tk_image = None

        self.image_item = self.canvas.create_image(
            0,
            0,
            anchor="nw"
        )

        # =========================================================
        # BRUSH CIRCLE
        # =========================================================

        self.brush_circle = self.canvas.create_oval(
            0,
            0,
            0,
            0,
            outline="white",
            width=2
        )

        # =========================================================
        # WINDOW RESIZE
        # =========================================================

        self.canvas.bind(
            "<Configure>",
            self.on_canvas_resize
        )

        # =========================================================
        # FIRST RENDER
        # =========================================================

        self.update_info()
        self.refresh_image()

    # =============================================================
    # SECTION HEADER
    # =============================================================

    def add_section(
        self,
        parent,
        text
    ):
        frame = tk.Frame(
            parent,
            height=35
        )

        frame.pack(
            fill="x",
            padx=8,
            pady=(12, 4)
        )

        label = tk.Label(
            frame,
            text=text,
            font=("Arial", 11, "bold")
        )

        label.pack(
            side="left"
        )

    # =============================================================
    # SLIDER
    # =============================================================

    def add_slider(
        self,
        parent,
        name,
        value,
        minimum,
        maximum,
        resolution,
        callback
    ):

        frame = tk.Frame(
            parent
        )

        frame.pack(
            fill="x",
            padx=10,
            pady=2
        )

        label = tk.Label(
            frame,
            text=name,
            width=18,
            anchor="w"
        )

        label.pack(
            side="left"
        )

        scale = tk.Scale(
            frame,
            from_=minimum,
            to=maximum,
            resolution=resolution,
            orient="horizontal",
            showvalue=True,
            command=callback,
            length=170
        )

        scale.set(value)

        scale.pack(
            side="left",
            fill="x",
            expand=True
        )

    # =============================================================
    # CANVAS RESIZE
    # =============================================================

    def on_canvas_resize(self, event):
        self.canvas.config(
            scrollregion=(
                0,
                0,
                self.work_w,
                self.work_h
            )
        )

    # =============================================================
    # INFO
    # =============================================================

    def update_info(self):

        if self.texture_path is None:
            texture_name = "None"
        else:
            texture_name = str(
                self.texture_path
            )

            if len(texture_name) > 32:
                texture_name = (
                    "..." + texture_name[-29:]
                )

        self.info.config(
            text=(
                f"Brush: {int(self.radius)}\n"
                f"Strength: {self.strength:.2f}\n"
                f"Texture: {texture_name}\n"
                f"Preview: "
                f"{self.work_w} × {self.work_h}"
            )
        )

    # =============================================================
    # SETTINGS
    # =============================================================

    def change_radius(self, value):
        self.radius += float(value)

        self.radius = max(
            5,
            min(
                500,
                self.radius
            )
        )

        self.update_info()

    def change_strength(self, value):
        self.strength = float(value)
        self.update_info()

    def change_feedback(self, value):
        self.feedback = float(value)
        self.refresh_image()

    def change_steps(self, value):
        self.feedback_steps = max(
            1,
            int(float(value))
        )

        self.refresh_image()

    def change_fold(self, value):
        self.fold = float(value)
        self.refresh_image()

    def change_twist(self, value):
        self.twist = float(value)
        self.refresh_image()

    def change_echo(self, value):
        self.echo_shift = float(value)
        self.refresh_image()

    def change_texture_size(self, value):

        size = int(float(value))

        if size % 2 == 0:
            size += 1

        self.texture_size = max(
            3,
            min(
                51,
                size
            )
        )

        if (
            self.texture_path is not None
            and self.texture_path != "RANDOM"
        ):
            try:
                self.build_texture_kernel(
                    self.texture_path
                )
            except Exception:
                pass

        self.refresh_image()

    def change_texture_strength(self, value):
        self.texture_strength = float(value)
        self.refresh_image()

    # =============================================================
    # LOAD TEXTURE
    # =============================================================

    def load_texture(self):

        path = filedialog.askopenfilename(
            title="Выберите текстуру",
            filetypes=[
                (
                    "Images",
                    "*.avif *.png *.jpg *.jpeg *.webp *.svg"
                ),
                ("AVIF", "*.avif"),
                ("PNG", "*.png"),
                ("JPEG", "*.jpg *.jpeg"),
                ("WEBP", "*.webp"),
                ("SVG", "*.svg"),
                ("All files", "*.*")
            ]
        )

        if not path:
            return

        try:

            self.build_texture_kernel(
                path
            )

            self.texture_path = path

            self.update_info()
            self.refresh_image()

        except Exception as e:

            messagebox.showerror(
                "Ошибка загрузки текстуры",
                f"{e}"
            )

    # =============================================================
    # BUILD TEXTURE KERNEL
    # =============================================================

    def build_texture_kernel(
        self,
        path
    ):

        size = int(
            self.texture_size
        )

        if size % 2 == 0:
            size += 1

        # ---------------------------------------------------------
        # SVG
        # ---------------------------------------------------------

        if path.lower().endswith(
            ".svg"
        ):

            import cairosvg

            png_data = cairosvg.svg2png(
                url=path,
                output_width=size,
                output_height=size
            )

            texture = Image.open(
                io.BytesIO(
                    png_data
                )
            ).convert("L")

        # ---------------------------------------------------------
        # RASTER
        # ---------------------------------------------------------

        else:

            texture = Image.open(
                path
            ).convert("L")

        # ---------------------------------------------------------
        # RESIZE
        # ---------------------------------------------------------

        texture = texture.resize(
            (
                size,
                size
            ),
            Image.Resampling.LANCZOS
        )

        kernel = np.asarray(
            texture,
            dtype=np.float32
        )

        # ---------------------------------------------------------
        # REMOVE DC
        # ---------------------------------------------------------

        kernel -= kernel.mean()

        # ---------------------------------------------------------
        # STD NORMALIZATION
        # ---------------------------------------------------------

        std = kernel.std()

        if std > 1e-6:
            kernel /= std

        # ---------------------------------------------------------
        # L1 NORMALIZATION
        # ---------------------------------------------------------

        absolute_sum = np.sum(
            np.abs(kernel)
        )

        if absolute_sum > 1e-6:
            kernel /= absolute_sum

        # ---------------------------------------------------------
        # FLIP FOR CONVOLUTION
        # ---------------------------------------------------------

        kernel = np.flip(
            kernel,
            axis=(0, 1)
        ).copy()

        self.texture_kernel = kernel

    # =============================================================
    # RANDOM KERNEL
    # =============================================================

    def randomize_kernel(self):

        size = int(
            self.texture_size
        )

        if size % 2 == 0:
            size += 1

        kernel = np.random.normal(
            0,
            1,
            (
                size,
                size
            )
        ).astype(
            np.float32
        )

        center = size // 2

        kernel[
            center,
            center
        ] += 1.5

        kernel -= kernel.mean()

        absolute_sum = np.sum(
            np.abs(kernel)
        )

        if absolute_sum > 1e-6:
            kernel /= absolute_sum

        self.texture_kernel = kernel
        self.texture_path = "RANDOM"

        self.update_info()
        self.refresh_image()

    # =============================================================
    # TEXTURE CONVOLUTION
    # =============================================================

    def apply_texture_convolution(
        self,
        image
    ):

        if self.texture_kernel is None:
            return image

        kernel = (
            self.texture_kernel
            * self.texture_strength
        )

        image_float = image.astype(
            np.float32
        )

        filtered = cv2.filter2D(
            image_float,
            ddepth=cv2.CV_32F,
            kernel=kernel,
            borderType=cv2.BORDER_REFLECT
        )

        result = (
            image_float
            + filtered
        )

        result = np.clip(
            result,
            0,
            255
        )

        return result.astype(
            np.uint8
        )

    # =============================================================
    # LIQUIFY
    # =============================================================

    def render_liquify(self):

        h, w = self.base_np.shape[:2]

        y, x = np.indices(
            (
                h,
                w
            ),
            dtype=np.float32
        )

        map_x = (
            x
            - self.disp_x
        )

        map_y = (
            y
            - self.disp_y
        )

        map_x = np.clip(
            map_x,
            0,
            w - 1
        )

        map_y = np.clip(
            map_y,
            0,
            h - 1
        )

        result = cv2.remap(
            self.base_np,
            map_x,
            map_y,
            interpolation=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT
        )

        return result

    # =============================================================
    # CHAOS ENGINE
    # =============================================================

    def render_chaos(self):

        current = self.render_liquify()

        h, w = current.shape[:2]

        y, x = np.indices(
            (
                h,
                w
            ),
            dtype=np.float32
        )

        cx = w / 2.0
        cy = h / 2.0

        rx = x - cx
        ry = y - cy

        distance = np.sqrt(
            rx * rx
            +
            ry * ry
        )

        max_radius = np.sqrt(
            cx * cx
            +
            cy * cy
        )

        normalized_radius = (
            distance
            /
            max_radius
        )

        # =========================================================
        # FEEDBACK
        # =========================================================

        for i in range(
            self.feedback_steps
        ):

            # -----------------------------------------------------
            # FOLD
            # -----------------------------------------------------

            period = max(
                30.0,
                min(
                    h,
                    w
                ) * 0.35
            )

            folded_x = (
                (
                    rx
                    +
                    period / 2
                )
                %
                period
            ) - period / 2

            folded_y = (
                (
                    ry
                    +
                    period / 2
                )
                %
                period
            ) - period / 2

            folded_x = np.abs(
                folded_x
            )

            folded_y = np.abs(
                folded_y
            )

            fold_mask = np.abs(
                np.sin(
                    normalized_radius
                    * np.pi
                )
            )

            fold_strength = (
                self.fold
                * fold_mask
            )

            fx = (
                rx
                * (1.0 - fold_strength)
                +
                folded_x
                * fold_strength
            )

            fy = (
                ry
                * (1.0 - fold_strength)
                +
                folded_y
                * fold_strength
            )

            # -----------------------------------------------------
            # TWIST
            # -----------------------------------------------------

            angle = (
                self.twist
                *
                np.sin(
                    normalized_radius
                    * np.pi
                    * 2.0
                )
                *
                (
                    1.0
                    -
                    (
                        i
                        /
                        max(
                            1,
                            self.feedback_steps
                        )
                    )
                )
            )

            cos_a = np.cos(
                angle
            )

            sin_a = np.sin(
                angle
            )

            tx = (
                fx * cos_a
                -
                fy * sin_a
            )

            ty = (
                fx * sin_a
                +
                fy * cos_a
            )

            # -----------------------------------------------------
            # ECHO
            # -----------------------------------------------------

            direction = (
                1.0
                if i % 2 == 0
                else -1.0
            )

            echo_x = (
                direction
                *
                self.echo_shift
                *
                (i + 1)
            )

            echo_y = (
                -direction
                *
                self.echo_shift
                *
                0.5
                *
                (i + 1)
            )

            src_x = (
                cx
                +
                tx
                +
                echo_x
            )

            src_y = (
                cy
                +
                ty
                +
                echo_y
            )

            src_x = np.clip(
                src_x,
                0,
                w - 1
            )

            src_y = np.clip(
                src_y,
                0,
                h - 1
            )

            warped = cv2.remap(
                current,
                src_x.astype(
                    np.float32
                ),
                src_y.astype(
                    np.float32
                ),
                interpolation=cv2.INTER_LINEAR,
                borderMode=cv2.BORDER_REFLECT
            )

            # -----------------------------------------------------
            # FEEDBACK MIX
            # -----------------------------------------------------

            alpha = (
                self.feedback
                *
                (
                    self.echo_decay
                    ** i
                )
            )

            alpha = np.clip(
                alpha,
                0.0,
                0.95
            )

            current = (
                current.astype(
                    np.float32
                )
                *
                (1.0 - alpha)
                +
                warped.astype(
                    np.float32
                )
                *
                alpha
            )

            current = np.clip(
                current,
                0,
                255
            ).astype(
                np.uint8
            )

            # -----------------------------------------------------
            # TEXTURE CONVOLUTION
            # -----------------------------------------------------

            if (
                self.texture_kernel
                is not None
            ):

                current = (
                    self.apply_texture_convolution(
                        current
                    )
                )

        return current

    # =============================================================
    # REFRESH IMAGE
    # =============================================================

    def refresh_image(self):

        result = self.render_chaos()

        pil_image = Image.fromarray(
            result
        )

        self.tk_image = ImageTk.PhotoImage(
            pil_image
        )

        self.canvas.itemconfig(
            self.image_item,
            image=self.tk_image
        )

        self.canvas.tag_raise(
            self.brush_circle
        )

    # =============================================================
    # MOUSE POSITION
    # =============================================================

    def mouse_position(
        self,
        event
    ):

        x = max(
            0,
            min(
                self.work_w - 1,
                event.x
            )
        )

        y = max(
            0,
            min(
                self.work_h - 1,
                event.y
            )
        )

        return (
            float(x),
            float(y)
        )

    # =============================================================
    # BRUSH
    # =============================================================

    def apply_brush(
        self,
        center_x,
        center_y,
        move_x,
        move_y
    ):

        radius = float(
            self.radius
        )

        x0 = max(
            0,
            int(
                center_x
                -
                radius
            )
        )

        x1 = min(
            self.work_w,
            int(
                center_x
                +
                radius
                +
                1
            )
        )

        y0 = max(
            0,
            int(
                center_y
                -
                radius
            )
        )

        y1 = min(
            self.work_h,
            int(
                center_y
                +
                radius
                +
                1
            )
        )

        if (
            x0 >= x1
            or
            y0 >= y1
        ):
            return

        yy, xx = np.mgrid[
            y0:y1,
            x0:x1
        ]

        dx = (
            xx
            -
            center_x
        )

        dy = (
            yy
            -
            center_y
        )

        distance = np.sqrt(
            dx * dx
            +
            dy * dy
        )

        mask = (
            distance
            <=
            radius
        )

        t = np.clip(
            1.0
            -
            distance / radius,
            0.0,
            1.0
        )

        weight = (
            t
            *
            t
            *
            (
                3.0
                -
                2.0 * t
            )
        )

        weight *= mask
        weight *= self.strength

        self.disp_x[
            y0:y1,
            x0:x1
        ] += (
            move_x
            *
            weight
        )

        self.disp_y[
            y0:y1,
            x0:x1
        ] += (
            move_y
            *
            weight
        )

    # =============================================================
    # BRUSH STROKE
    # =============================================================

    def apply_stroke(
        self,
        x0,
        y0,
        x1,
        y1
    ):

        dx = x1 - x0
        dy = y1 - y0

        distance = np.sqrt(
            dx * dx
            +
            dy * dy
        )

        if distance < 0.01:
            return

        step_size = max(
            2.0,
            self.radius * 0.15
        )

        steps = max(
            1,
            int(
                distance
                /
                step_size
            )
        )

        for i in range(
            1,
            steps + 1
        ):

            t = i / steps

            px = (
                x0
                +
                dx * t
            )

            py = (
                y0
                +
                dy * t
            )

            self.apply_brush(
                px,
                py,
                dx / steps,
                dy / steps
            )

    # =============================================================
    # MOUSE DOWN
    # =============================================================

    def mouse_down(
        self,
        event
    ):

        (
            self.last_x,
            self.last_y
        ) = self.mouse_position(
            event
        )

    # =============================================================
    # MOUSE DRAG
    # =============================================================

    def mouse_drag(
        self,
        event
    ):

        x, y = self.mouse_position(
            event
        )

        if (
            self.last_x is not None
            and
            self.last_y is not None
        ):

            self.apply_stroke(
                self.last_x,
                self.last_y,
                x,
                y
            )

            self.refresh_image()

        self.last_x = x
        self.last_y = y

    # =============================================================
    # MOUSE UP
    # =============================================================

    def mouse_up(
        self,
        event
    ):

        self.last_x = None
        self.last_y = None

    # =============================================================
    # MOUSE MOVE
    # =============================================================

    def mouse_move(
        self,
        event
    ):

        x, y = self.mouse_position(
            event
        )

        r = self.radius

        self.canvas.coords(
            self.brush_circle,
            x - r,
            y - r,
            x + r,
            y + r
        )

        self.canvas.tag_raise(
            self.brush_circle
        )

    # =============================================================
    # MOUSE WHEEL
    # =============================================================

    def mouse_wheel(
        self,
        event
    ):

        if event.delta > 0:
            self.change_radius(10)
        else:
            self.change_radius(-10)

    # =============================================================
    # RANDOM CHAOS
    # =============================================================

    def random_chaos(
        self
    ):

        self.feedback = np.random.uniform(
            0.25,
            0.90
        )

        self.feedback_steps = np.random.randint(
            2,
            9
        )

        self.fold = np.random.uniform(
            0.0,
            0.95
        )

        self.twist = np.random.uniform(
            -1.5,
            1.5
        )

        self.echo_shift = np.random.uniform(
            0,
            35
        )

        self.strength = np.random.uniform(
            0.3,
            1.5
        )

        self.texture_strength = np.random.uniform(
            0.0,
            1.0
        )

        self.update_info()
        self.refresh_image()

    # =============================================================
    # RESET
    # =============================================================

    def reset(
        self
    ):

        self.disp_x.fill(0)
        self.disp_y.fill(0)

        self.feedback = 0.45
        self.feedback_steps = 4

        self.fold = 0.15
        self.twist = 0.15

        self.echo_shift = 5.0

        self.strength = 0.8
        self.texture_strength = 0.35

        self.update_info()
        self.refresh_image()

    # =============================================================
    # SAVE
    # =============================================================

    def save(
        self
    ):

        output = filedialog.asksaveasfilename(
            title="Сохранить",
            defaultextension=".avif",
            filetypes=[
                ("AVIF", "*.avif"),
                ("PNG", "*.png"),
                ("JPEG", "*.jpg"),
            ]
        )

        if not output:
            return

        try:

            result = self.render_chaos()

            # Возвращаем к оригинальному размеру
            if (
                result.shape[1]
                != self.original_w
                or
                result.shape[0]
                != self.original_h
            ):

                result_image = Image.fromarray(
                    result
                )

                result_image = (
                    result_image.resize(
                        (
                            self.original_w,
                            self.original_h
                        ),
                        Image.Resampling.LANCZOS
                    )
                )

            else:

                result_image = Image.fromarray(
                    result
                )

            lower = output.lower()

            if lower.endswith(
                ".avif"
            ):

                result_image.save(
                    output,
                    format="AVIF",
                    quality=90
                )

            elif (
                lower.endswith(".jpg")
                or
                lower.endswith(".jpeg")
            ):

                result_image.save(
                    output,
                    format="JPEG",
                    quality=95
                )

            else:

                result_image.save(
                    output,
                    format="PNG"
                )

            messagebox.showinfo(
                "Готово",
                f"Изображение сохранено:\n\n{output}"
            )

        except Exception as e:

            messagebox.showerror(
                "Ошибка сохранения",
                str(e)
            )


# ================================================================
# START
# ================================================================

if __name__ == "__main__":

    root = tk.Tk()

    app = ChaosImageApp(
        root
    )

    root.mainloop()